# Research Question and Provisional Lane

This is my starting question for the project. I have chosen a lane based on a quick look at the starter data, and I can change it until the end of Week 4.

## 1. My lane (or freestyle) and why

I am choosing **Refresh / Content Opportunity Scoring (Lane 2)**. My aim is to help an editor decide which pages to look at first when there are too many to review at once.

The starter data includes search impressions, recent changes in impressions, and time since the last update. These give me a place to start. My initial check shows that many pages have declining impressions, while only a few match the rule for older pages that still get visibility. I want to explore whether looking at these signals together gives a more useful review list.

Over the next seven weeks, I plan to understand the data, try a simple ranking method, and check whether a more detailed approach adds value. I am keeping the choice provisional because I still need to learn what the data can support.

In [1]:
LANE = "Refresh / Content Opportunity Scoring"
REVIEW_BUDGET = 20  # Provisional pages reviewed per client per cycle.
MIN_DEMAND_IMPRESSIONS = 100
STALE_DAYS = 180
STALE_VISIBLE_IMPRESSIONS = 500
print(f"Provisional lane: {LANE}")
print(f"Provisional review budget: {REVIEW_BUDGET} pages per client per cycle")

Provisional lane: Refresh / Content Opportunity Scoring
Provisional review budget: 20 pages per client per cycle


## 2. The question: decision, action, cost of a wrong call

**My question:** Which pages should an editor review first for a possible refresh, and what information helps make that choice?

**What one row means:** One page belonging to one client. The starter file is a snapshot, so I am comparing pages at the same point in the dataset, rather than following them over time.

**The decision and output:** I want to create a ranked list of pages to review, with a short reason for each suggestion. I will start with up to 20 pages per client as a working review limit. That number can change depending on the editor's available time.

**Who uses it and what they do:** A content editor would look at the suggested pages, check the content and search context, and decide whether to update the page or keep watching it. A score should help them choose where to look, rather than make the editing decision for them.

**What a wrong suggestion costs:** Recommending a page that does not need attention wastes time and could lead to unnecessary changes. Missing a page that does need attention could delay a useful update while its visibility continues to fall. This dataset does not tell me the financial cost of either mistake.

**Why data or ML could help:** Reviewing thousands of pages manually is difficult. Data can help narrow the list using signals such as demand, recent changes, and update age. I will first compare a simple rule with a list ordered by impressions. I would only use ML if it gives a useful improvement on data that was not used to build it.

**How I will check it:** This is mainly a ranking problem. For now, I can check how many of the top 20 pages have the dataset's existing decline flag. This is called precision@20, but here it is only a check against a recent trend, not proof that those pages need editing. I will compare results within each client and report smaller lists separately.

If I later use the daily warehouse data, I want to check whether the ranking identifies pages whose impressions fall by more than 20% in the following 30 days compared with the preceding 30 days. I would start with pages that had at least 100 impressions in the preceding period, use only information available before the decision, and test on later periods and clients kept aside. The windows and thresholds still need checking. If I cannot do that reliably, I will keep the project focused on describing the data and supporting review decisions.

In [2]:
FRAME = {
    "unit": "page within client at a decision date",
    "actor": "content editor",
    "decision": "which pages to review first",
    "action": "inspect, then decide whether to refresh or monitor",
    "metric": "precision@20 against an explicitly named decline proxy",
}
assert REVIEW_BUDGET > 0 and MIN_DEMAND_IMPRESSIONS > 0
assert all(FRAME.values())
print("Decision, actor, action, grain, and evaluation metric are recorded.")

Decision, actor, action, grain, and evaluation metric are recorded.


## 3. Quick look at the data (2-3 real numbers)

I loaded the starter CSV and checked three things: how many pages it contains, how many have declining impressions and some search demand, and how many have gone a long time without an update while still getting visibility.

For this first check, I used at least **100 impressions over 90 days** as a basic demand filter. For the older-page rule, I used **180 days since the last update** and at least **500 impressions over 90 days**, following the starter example. These are starting thresholds, not proven cutoffs.

In [3]:
from pathlib import Path
import pandas as pd

# Locate the CSV from the repository root or a nested notebook directory.
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data/raw/content_refresh_anonymized.csv").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook inside a checkout containing the starter CSV.")
data = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")
required = {"content_id", "client_id", "impressions_90d", "trend_direction",
            "days_since_last_update", "content_age_days", "avg_position", "ctr"}
assert required.issubset(data.columns), "Required starter columns are missing."
assert not data[["client_id", "content_id"]].isna().any().any()
assert data["content_id"].is_unique, "Check page grain before counting."
assert len(data) > 0

has_demand = data["impressions_90d"].ge(MIN_DEMAND_IMPRESSIONS)
declining_with_demand = has_demand & data["trend_direction"].eq("down")
stale_visible = (
    data["days_since_last_update"].ge(STALE_DAYS)
    & data["impressions_90d"].ge(STALE_VISIBLE_IMPRESSIONS)
)
print(f"1. Inventory: {len(data):,} unique pages, {data.client_id.nunique():,} clients, "
      f"{len(data.columns)} columns; duplicate page IDs: 0.")
print(f"2. Declining with demand: {declining_with_demand.sum():,} / {len(data):,} pages "
      f"({100 * declining_with_demand.mean():.2f}%).")
print(f"3. Stale and visible: {stale_visible.sum():,} / {len(data):,} pages "
      f"({100 * stale_visible.mean():.2f}%).")

1. Inventory: 30,000 unique pages, 32 clients, 44 columns; duplicate page IDs: 0.
2. Declining with demand: 13,152 / 30,000 pages (43.84%).
3. Stale and visible: 17 / 30,000 pages (0.06%).


**What I found:**

- The file contains **30,000 unique pages across 32 clients**.
- **13,152 pages (43.84%)** have the decline flag and at least 100 impressions over 90 days.
- Only **17 pages (0.06%)** meet the older-page rule above.

These numbers make this lane worth exploring because there are many pages that could be reviewed, but update age alone picks out very few. I want to find a sensible way to narrow that larger group down.

The decline flag compares impressions in the last 30 days with the previous 30 days. It does not tell me what happens next. The two groups can overlap, and meeting either rule does not automatically mean a page needs a refresh. Also, 100 impressions over 90 days can still leave very little data in a particular 30-day period.

## 4. Careful words: what I can and can't claim

I can describe what I **observed in this dataset** and suggest pages that may deserve a closer look. I cannot say that updating those pages will improve traffic, that a decline was caused by old content, or that this analysis explains Google's ranking system. I also cannot assume this starter sample represents every client's full inventory.

The existing decline flag is calculated from the recent impression change. If I train a model to reproduce it, that would not prove I can predict future decline. I would need to keep `trend_direction`, `trend_pct`, and the impression fields used to calculate that label out of the model's inputs. Even the 90-day totals include part of the same period, so a future prediction needs separate earlier and later windows. When using the warehouse, I would check each client's available history and leave the final month untouched until the final test.

I also need to read the measurements carefully: IDs are for grouping rather than prediction, position zero means missing data, and rate values are already percentages. Missing values should not automatically become zeros, especially where tracking was unavailable.

**A simple starting comparison:** The next cell picks pages with at least 100 impressions, orders them by impressions within each client, and takes up to 20. It then checks how many have the current decline flag. The flag is not used to choose the pages. The average is calculated across clients with a full list of 20, while shorter lists are reported separately. This gives me a starting point to compare with later, rather than a tested recommendation system.

In [4]:
eligible = data.loc[has_demand].copy()
# ID is only a deterministic tie break, not a predictive feature.
queue = (
    eligible.sort_values(["client_id", "impressions_90d", "content_id"],
                         ascending=[True, False, True])
    .groupby("client_id", sort=False).head(REVIEW_BUDGET)
)
queue = queue.assign(observed_down=queue["trend_direction"].eq("down"))
per_client = queue.groupby("client_id")["observed_down"].agg(["mean", "size"])
full_queues = per_client.loc[per_client["size"].eq(REVIEW_BUDGET)]
short_queues = per_client.loc[per_client["size"].lt(REVIEW_BUDGET)]
assert not full_queues.empty, "Not enough eligible pages for precision@20."
print(f"Demand-only baseline: {len(queue):,} selected pages across {len(per_client)} clients.")
print(f"Current-proxy macro precision@{REVIEW_BUDGET}: "
      f"{100 * full_queues['mean'].mean():.2f}% "
      f"({len(full_queues)} clients with full queues).")
print(f"Short queues reported separately: {len(short_queues)} clients.")
if not short_queues.empty:
    print(f"Short-queue mean current decline share: {100 * short_queues['mean'].mean():.2f}%.")
base_rates = (
    eligible.assign(observed_down=eligible["trend_direction"].eq("down"))
    .groupby("client_id")["observed_down"].mean()
)
print(f"Eligible-pool macro current decline share (same full-queue clients): "
      f"{100 * base_rates.loc[full_queues.index].mean():.2f}%.")
print("Diagnostic only: a current trend flag, not future decline or refresh payoff.")

Demand-only baseline: 528 selected pages across 30 clients.
Current-proxy macro precision@20: 47.29% (24 clients with full queues).
Short queues reported separately: 6 clients.
Short-queue mean current decline share: 41.94%.
Eligible-pool macro current decline share (same full-queue clients): 58.66%.
Diagnostic only: a current trend flag, not future decline or refresh payoff.


## 5. Self-check

- [x] I chose a lane and explained why I want to explore it.
- [x] I named the question, what one row means, who would use the result, and what they could do.
- [x] I explained the cost of getting a suggestion wrong.
- [x] I loaded the starter data and included three real measurements.
- [x] I explained why I will start simple and only use ML if it helps.
- [x] I separated what the data shows from what I still need to test.
- [x] The notebook runs from top to bottom and has saved outputs.
- [x] I included only aggregate results, with no private names, page URLs, queries, or credentials.
- [x] The notebook is saved and committed under `work/notebooks/`.

My repository: [FlyrankAI-Internship](https://github.com/vibhanshu-s/FlyrankAI-Internship).